# SEC EDGAR Raw Outputs Exploration

This notebook shows **exactly what SEC gives us** vs **what BNY needs**, in four layers:

| Layer | Source | Structured BNY fields? |
|-------|--------|------------------------|
| **A. EFTS JSON alone** | `efts.sec.gov/LATEST/search-index` | ~10–15% (ids, names, form, date) |
| **B. Filing HTML / exhibits** | `www.sec.gov/Archives/edgar/data/...` | Almost all *public* material facts (as text) |
| **C. Target extraction JSON** | Our layer (not from SEC) | Full public schema + evidence |
| **D. Client notification JSON** | Our layer + BNY placeholders | Public fields + internal nulls |

We make live API calls where possible and fall back to cached sample docs.


In [1]:
from pathlib import Path
import sys, json, re, textwrap
from IPython.display import display, Markdown, HTML

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
from eda.sec_client import SecClient
from eda.documents import html_to_text

client = SecClient()
pd.set_option('display.max_colwidth', 120)
print('ROOT =', ROOT)


ROOT = /Users/kedarvaidya/Desktop/BNY-Capstone-Project


## A. EFTS JSON alone — live API call

Endpoint:
```text
GET https://efts.sec.gov/LATEST/search-index?q=...&forms=SC TO-T&startdt=...&enddt=...&from=0&size=3
```


In [2]:
efts = client.efts_search({
    'q': 'a OR the OR of',
    'dateRange': 'custom',
    'startdt': '2020-01-01',
    'enddt': '2020-01-31',
    'forms': 'SC TO-T',
    'from': 0,
    'size': 3,
})

print('Top-level keys:', list(efts.keys()))
print('Total hits in window:', efts.get('hits', {}).get('total'))
print()
print(json.dumps({
    'took': efts.get('took'),
    'hits': {
        'total': efts.get('hits', {}).get('total'),
        'hits': efts.get('hits', {}).get('hits', [])[:1],  # one full raw hit
    }
}, indent=2)[:6000])


Top-level keys: ['took', 'timed_out', '_shards', 'hits', 'aggregations', 'query']
Total hits in window: {'value': 78, 'relation': 'eq'}

{
  "took": 122,
  "hits": {
    "total": {
      "value": 78,
      "relation": "eq"
    },
    "hits": [
      {
        "_index": "edgar_file",
        "_id": "0001193125-20-011457:d859616dex99a1b.htm",
        "_score": 0.7228609,
        "_source": {
          "ciks": [
            "0001557883",
            "0000059478"
          ],
          "period_ending": null,
          "file_num": [
            "005-88347"
          ],
          "display_names": [
            "Dermira, Inc.  (CIK 0001557883)",
            "ELI LILLY & Co  (LLY)  (CIK 0000059478)"
          ],
          "xsl": null,
          "sequence": 3,
          "root_forms": [
            "SC TO-T"
          ],
          "file_date": "2020-01-22",
          "biz_states": [
            "CA",
            "IN"
          ],
          "sics": [
            "2834",
            "2834"
       

### What EFTS covers on the BNY checklist


In [3]:
# Pick first hit with a usable CIK + accession
raw_hits = efts.get('hits', {}).get('hits', [])
assert raw_hits, 'No EFTS hits returned — check network / User-Agent'
hit = raw_hits[0]
src = hit['_source']

# Prefer a primary form doc if present in the 3 hits
for h in raw_hits:
    if h['_source'].get('file_type') in {h['_source'].get('form'), 'SC TO-T', 'SC TO-T/A'}:
        hit, src = h, h['_source']
        break

EXAMPLE = {
    'efts_id': hit['_id'],
    'accession': src.get('adsh'),
    'form': src.get('form'),
    'file_date': src.get('file_date'),
    'file_type': src.get('file_type'),
    'file_num': src.get('file_num'),
    'ciks': src.get('ciks'),
    'display_names': src.get('display_names'),
    'filename_from_id': hit['_id'].split(':', 1)[1] if ':' in hit['_id'] else None,
}

checklist_efts = [
    ('event_id', 'PARTIAL', 'file_num → we use as event_id'),
    ('corporate_action_type', 'PARTIAL', 'form/root_forms ≈ tender only'),
    ('event_status', 'PARTIAL', '/A in form name ≈ amended; not completed/withdrawn'),
    ('mandatory_voluntary_indicator', 'NO', 'not in EFTS JSON'),
    ('parties', 'PARTIAL', 'display_names / ciks — no roles'),
    ('security_description', 'WEAK', 'entity name only'),
    ('isin / cusip / ticker', 'WEAK', 'ticker sometimes inside display_names'),
    ('currency', 'NO', ''),
    ('available_options / default_option', 'NO', ''),
    ('effective / election / expiration / settlement dates', 'NO', ''),
    ('financial_terms', 'NO', ''),
    ('conditions_restrictions', 'NO', ''),
    ('source_document', 'YES', 'adsh + filename in _id'),
    ('source_passages / confidence / exceptions', 'NO', 'our layer'),
]
display(pd.DataFrame(checklist_efts, columns=['field', 'in_EFTS_JSON', 'note']))
print('\nExample hit summary:')
print(json.dumps(EXAMPLE, indent=2))


,field,in_EFTS_JSON,note
0,event_id,PARTIAL,file_num → we use as event_id
1,corporate_action_type,PARTIAL,form/root_forms ≈ tender only
2,event_status,PARTIAL,/A in form name ≈ amended; not completed/withdrawn
3,mandatory_voluntary_indicator,NO,not in EFTS JSON
4,parties,PARTIAL,display_names / ciks — no roles
5,security_description,WEAK,entity name only
6,isin / cusip / ticker,WEAK,ticker sometimes inside display_names
7,currency,NO,
8,available_options / default_option,NO,
9,effective / election / expiration / settlement dates,NO,



Example hit summary:
{
  "efts_id": "0001193125-20-011114:d874751dsctota.htm",
  "accession": "0001193125-20-011114",
  "form": "SC TO-T/A",
  "file_date": "2020-01-21",
  "file_type": "SC TO-T/A",
  "file_num": [
    "005-86024"
  ],
  "ciks": [
    "0001492658",
    "0000310764"
  ],
  "display_names": [
    "Wright Medical Group N.V.  (CIK 0001492658)",
    "STRYKER CORP  (SYK)  (CIK 0000310764)"
  ],
  "filename_from_id": "d874751dsctota.htm"
}


## B. Filing HTML / exhibits — raw EDGAR Archives

### B1. Filing index JSON (list of documents in the accession)
```text
GET https://www.sec.gov/Archives/edgar/data/{cik}/{accession_nodash}/index.json
```


In [4]:
cik = str(src['ciks'][0])
accession = src['adsh']
filename = EXAMPLE['filename_from_id']

try:
    index = client.filing_index(cik, accession)
    print('index.json keys:', list(index.keys()))
    items = index.get('directory', {}).get('item', [])
    # normalize single item
    if isinstance(items, dict):
        items = [items]
    print(f'\nFiles in accession {accession}: {len(items)}')
    display(pd.DataFrame(items)[['name', 'type', 'size']].head(20) if items else pd.DataFrame())
    print('\nRaw index.json (truncated):')
    print(json.dumps(index, indent=2)[:3500])
except Exception as exc:
    index, items = {}, []
    print('index.json fetch failed:', exc)
    print('Continuing with document download using EFTS filename.')


index.json keys: ['directory']

Files in accession 0001193125-20-011114: 5


,name,type,size
0,0001193125-20-011114-index-headers.html,text.gif,
1,0001193125-20-011114-index.html,text.gif,
2,0001193125-20-011114.txt,text.gif,
3,d874751dsctota.htm,text.gif,51789
4,filename2.htm,text.gif,51023



Raw index.json (truncated):
{
  "directory": {
    "item": [
      {
        "last-modified": "2020-01-21 17:05:13",
        "name": "0001193125-20-011114-index-headers.html",
        "type": "text.gif",
        "size": ""
      },
      {
        "last-modified": "2020-01-21 17:05:13",
        "name": "0001193125-20-011114-index.html",
        "type": "text.gif",
        "size": ""
      },
      {
        "last-modified": "2020-01-21 17:05:13",
        "name": "0001193125-20-011114.txt",
        "type": "text.gif",
        "size": ""
      },
      {
        "last-modified": "2020-01-21 17:05:13",
        "name": "d874751dsctota.htm",
        "type": "text.gif",
        "size": "51789"
      },
      {
        "last-modified": "2020-01-21 17:05:13",
        "name": "filename2.htm",
        "type": "text.gif",
        "size": "51023"
      }
    ],
    "name": "/Archives/edgar/data/1492658/000119312520011114",
    "parent-dir": "/Archives/edgar/data/1492658"
  }
}


### B2. Primary filing HTML (raw bytes / text excerpt)
```text
GET https://www.sec.gov/Archives/edgar/data/{cik}/{accession_nodash}/{filename}
```


In [5]:
url = client.document_url(cik, accession, filename)
print('Document URL:', url)

cache_dir = ROOT / 'data' / 'corpus' / 'sample_docs'
cache_dir.mkdir(parents=True, exist_ok=True)
local = cache_dir / f"{accession}__{filename.replace('/', '_')}"

try:
    if local.exists():
        raw = local.read_bytes()
        print('Loaded from cache:', local.name, f'({len(raw):,} bytes)')
    else:
        raw = client.get_bytes(url)
        local.write_bytes(raw)
        print('Downloaded:', local.name, f'({len(raw):,} bytes)')
    html = raw.decode('utf-8', errors='replace')
    text = html_to_text(html)
except Exception as exc:
    html, text = '', ''
    print('Download failed:', exc)
    # fallback to any cached SC TO-T sample
    fallbacks = sorted((ROOT / 'data/corpus/sample_docs').glob('*sctot*.htm'))
    if fallbacks:
        local = fallbacks[0]
        html = local.read_text(encoding='utf-8', errors='replace')
        text = html_to_text(html)
        print('Using fallback sample:', local.name)

print('\n--- RAW HTML head (first 1500 chars) ---\n')
print(html[:1500])
print('\n--- PLAIN TEXT head (first 2000 chars) ---\n')
print(text[:2000])


Document URL: https://www.sec.gov/Archives/edgar/data/1492658/000119312520011114/d874751dsctota.htm
Loaded from cache: 0001193125-20-011114__d874751dsctota.htm (51,789 bytes)

--- RAW HTML head (first 1500 chars) ---

<DOCUMENT>
<TYPE>SC TO-T/A
<SEQUENCE>1
<FILENAME>d874751dsctota.htm
<DESCRIPTION>SC TO-T/A
<TEXT>
<HTML><HEAD>
<TITLE>SC TO-T/A</TITLE>
</HEAD>
 <BODY BGCOLOR="WHITE">


<Center><DIV STYLE="width:8.5in" align="left">
 <P STYLE="line-height:1.0pt;margin-top:0pt;margin-bottom:0pt;border-bottom:1px solid #000000">&nbsp;</P>
<P STYLE="line-height:3.0pt;margin-top:0pt;margin-bottom:2pt;border-bottom:1px solid #000000">&nbsp;</P> <P STYLE="margin-top:6pt; margin-bottom:0pt; font-size:18pt; font-family:Times New Roman" ALIGN="center"><B>UNITED STATES </B></P>
<P STYLE="margin-top:0pt; margin-bottom:0pt; font-size:18pt; font-family:Times New Roman" ALIGN="center"><B>SECURITIES AND EXCHANGE COMMISSION </B></P>
<P STYLE="margin-top:0pt; margin-bottom:0pt; font-size:12pt; font-famil

### B3. Does the HTML *mention* the material facts? (keyword scan — not extraction)


In [6]:
PATTERNS = {
    'parties_offeror_target': r'offeror|bidder|purchaser|subject company|target company',
    'cusip': r'\bCUSIP\b',
    'isin': r'\bISIN\b',
    'ticker': r'ticker\s+symbol|trading\s+symbol',
    'offer_price / cash': r'offer\s+price|\$\s?\d+(?:\.\d+)?\s+per\s+share|cash\s+consideration',
    'expiration_date': r'expir(?:e|es|ation)\s+date|unless\s+extended',
    'withdrawal_deadline': r'right\s+to\s+withdraw|withdrawal\s+(?:rights?|deadline)',
    'election / options': r'\belection\b|cash\s+election|stock\s+election|mixed\s+consideration',
    'proration': r'proration|pro\s*ration',
    'minimum_condition': r'minimum\s+(?:tender\s+)?condition|minimum\s+tender',
    'conditions': r'conditions?\s+to\s+(?:the\s+)?offer|financing\s+condition|regulatory\s+approvals?',
    'restrictions': r'restricted\s+jurisdict|not\s+being\s+made',
    'exchange_ratio': r'exchange\s+ratio|conversion\s+ratio',
    'settlement / payment': r'settlement|promptly\s+after\s+expiration|payment\s+date',
    'voluntary / mandatory': r'\bvoluntary\b|\bmandatory\b',
    'amended / withdrawn / terminated': r'\bamended\b|\bwithdrawn\b|\bterminated\b|\bcancell?ed\b',
}

rows = []
for label, pat in PATTERNS.items():
    ms = list(re.finditer(pat, text, flags=re.I))
    snippet = ''
    if ms:
        m = ms[0]
        start = max(0, m.start() - 80)
        end = min(len(text), m.end() + 120)
        snippet = re.sub(r'\s+', ' ', text[start:end]).strip()
    rows.append({
        'material_fact': label,
        'mentions': len(ms),
        'in_this_filing_text': 'YES' if ms else 'NO',
        'example_passage': snippet[:220],
    })

cov = pd.DataFrame(rows)
display(cov)
print('\nMention coverage on this one filing:',
      f"{(cov['mentions']>0).mean():.0%} of scanned concepts")


,material_fact,mentions,in_this_filing_text,example_passage
0,parties_offeror_target,30,YES,Act of 1934 (Rule 14d-100) (Amendment No. 2) Wright Medical Group N.V. (Name of Subject Company (Issuer)) Stryker B....
1,cusip,1,YES,"ary shares, par value €0.03 per share (Title of Class of Securities) N96617118 (CUSIP Number of Class of Securities)..."
2,isin,0,NO,
3,ticker,0,NO,
4,offer_price / cash,6,YES,"er share, of Wright Medical Group N.V. multiplied by the offer consideration of $30.75 per share, (ii) the net offer..."
5,expiration_date,0,NO,
6,withdrawal_deadline,0,NO,
7,election / options,1,YES,"x, and other implications for Stryker, Purchaser and Wright associated with the election of each such Post-Offer Reo..."
8,proration,0,NO,
9,minimum_condition,10,YES,ing conditions are for the benefit of Stryker and Purchaser and (except for the Minimum Condition and the condition ...



Mention coverage on this one filing: 44% of scanned concepts


### B4. Optional: pull a related exhibit (Offer to Purchase / Letter of Transmittal) if listed


In [7]:
exhibit_candidates = []
for it in items if isinstance(items, list) else []:
    name = (it.get('name') or '').lower()
    typ = (it.get('type') or '').upper()
    if ('ex-99' in typ.lower() or 'ex99' in name.replace('.', '')
            or 'offer' in name or 'transmittal' in name):
        exhibit_candidates.append(it)

print(f'Exhibit-like files in index: {len(exhibit_candidates)}')
display(pd.DataFrame(exhibit_candidates)[['name','type','size']].head(10) if exhibit_candidates else pd.DataFrame({'note':['none or index unavailable']}))

if exhibit_candidates:
    ex = exhibit_candidates[0]
    ex_name = ex['name']
    ex_url = client.document_url(cik, accession, ex_name)
    print('Fetching exhibit:', ex_url)
    try:
        ex_raw = client.get_bytes(ex_url)
        ex_html = ex_raw.decode('utf-8', errors='replace')
        ex_text = html_to_text(ex_html)
        print(f'Exhibit size: {len(ex_raw):,} bytes')
        print('\n--- Exhibit text head ---\n')
        print(ex_text[:1800])
    except Exception as exc:
        print('Exhibit download failed:', exc)


Exhibit-like files in index: 0


,note
0,none or index unavailable


## C. Target structured JSON — **not returned by SEC**

This is the shape **we** must build from HTML (extraction + citations). Empty/nulls show the gap.


In [8]:
TARGET_EXTRACTION = {
    'event_id': (src.get('file_num') or [None])[0],
    'corporate_action_type': 'tender_offer',  # derived; SEC does not emit this label
    'event_status': None,  # announced|amended|extended|completed|withdrawn|cancelled
    'mandatory_voluntary_indicator': None,
    'parties': {
        'issuer': None,
        'target': None,
        'offeror': None,
        'acquirer': None,
        'raw_display_names': src.get('display_names'),  # only raw from EFTS
    },
    'security': {
        'security_description': None,
        'isin': None,
        'cusip': None,
        'ticker': None,
        'currency': None,
    },
    'client_decision': {
        'election_required': None,
        'available_options': [],
        'default_option': None,
    },
    'dates': {
        'record_date': None,
        'election_deadline': None,
        'expiration_date': None,
        'effective_date': None,
        'settlement_date': None,
        'withdrawal_deadline': None,
    },
    'financial_terms': {
        'consideration_type': None,
        'offer_price': None,
        'exchange_ratio': None,
        'conversion_ratio': None,
        'conversion_price': None,
        'subscription_price': None,
    },
    'conditions': {
        'eligibility': None,
        'minimum_tender_condition': None,
        'proration_terms': None,
        'withdrawal_rights': None,
        'restrictions': None,
    },
    'audit': {
        'source_document': {
            'accession': accession,
            'filename': filename,
            'url': url if 'url' in dir() else client.document_url(cik, accession, filename),
            'form': src.get('form'),
        },
        'fields': {
            # example of how EACH material field should look once extracted:
            'offer_price': {
                'value': None,
                'normalized_value': None,
                'source_passage': None,
                'confidence': None,
                'exception_flags': ['not_extracted_yet'],
            }
        }
    },
    'temporal': {
        'is_amendment': str(src.get('form', '')).endswith('/A'),
        'previous_value': None,
        'current_value': None,
        'change_source': None,
    },
    'event_specific': {
        'tender_offer': {
            'offer_price': None,
            'expiration_date': None,
            'withdrawal_deadline': None,
            'minimum_tender_condition': None,
            'proration_terms': None,
            'tendering_conditions': None,
        }
    },
}

print(json.dumps(TARGET_EXTRACTION, indent=2))
display(Markdown('**SEC does not return this object.** Filling the `null`s is the MVP extraction task.'))


{
  "event_id": "005-86024",
  "corporate_action_type": "tender_offer",
  "event_status": null,
  "mandatory_voluntary_indicator": null,
  "parties": {
    "issuer": null,
    "target": null,
    "offeror": null,
    "acquirer": null,
    "raw_display_names": [
      "Wright Medical Group N.V.  (CIK 0001492658)",
      "STRYKER CORP  (SYK)  (CIK 0000310764)"
    ]
  },
  "security": {
    "security_description": null,
    "isin": null,
    "cusip": null,
    "ticker": null,
    "currency": null
  },
  "client_decision": {
    "election_required": null,
    "available_options": [],
    "default_option": null
  },
  "dates": {
    "record_date": null,
    "election_deadline": null,
    "expiration_date": null,
    "effective_date": null,
    "settlement_date": null,
    "withdrawal_deadline": null
  },
  "financial_terms": {
    "consideration_type": null,
    "offer_price": null,
    "exchange_ratio": null,
    "conversion_ratio": null,
    "conversion_price": null,
    "subscription_pr

**SEC does not return this object.** Filling the `null`s is the MVP extraction task.

## D. Full client-notification JSON — public fields + **BNY internal placeholders**


In [9]:
NOTIFICATION_DRAFT = {
    'notification_family': 'Corporate Action Notification',
    'notification_type': 'ANNOUNCEMENT',  # BNY taxonomy — mapped by us
    'notification_status': 'DRAFT',
    'event_id': TARGET_EXTRACTION['event_id'],
    'message_id': None,              # BNY internal
    'bny_location': None,            # BNY internal
    'preparation_date': None,
    'sender': 'BNY Mellon (demo placeholder)',
    'recipient': None,               # BNY internal client
    'email_subject': None,

    # --- from public extraction (currently null until model runs) ---
    'corporate_action_type': TARGET_EXTRACTION['corporate_action_type'],
    'mandatory_voluntary_indicator': None,
    'security_description': None,
    'isin': None,
    'cusip': None,
    'ticker': None,
    'currency': None,
    'available_options': [],
    'default_option': None,
    'election_deadline': None,
    'expiration_date': None,
    'effective_date': None,
    'settlement_date': None,
    'offer_price': None,
    'conditions_restrictions': None,

    # --- BNY internal: never from SEC ---
    'processing_status': None,
    'account_number': None,
    'account_name': None,
    'eligible_quantity': None,
    'position_quantity': None,
    'cash_amount': None,            # client proceeds = price × position
    'debit_credit_indicator': None,

    'audit': {
        'source_documents': [TARGET_EXTRACTION['audit']['source_document']],
        'confidence_summary': None,
        'exception_flags': ['extraction_not_run', 'internal_fields_unpopulated'],
    }
}

print(json.dumps(NOTIFICATION_DRAFT, indent=2))


{
  "notification_family": "Corporate Action Notification",
  "notification_type": "ANNOUNCEMENT",
  "notification_status": "DRAFT",
  "event_id": "005-86024",
  "message_id": null,
  "bny_location": null,
  "preparation_date": null,
  "sender": "BNY Mellon (demo placeholder)",
  "recipient": null,
  "email_subject": null,
  "corporate_action_type": "tender_offer",
  "mandatory_voluntary_indicator": null,
  "security_description": null,
  "isin": null,
  "cusip": null,
  "ticker": null,
  "currency": null,
  "available_options": [],
  "default_option": null,
  "election_deadline": null,
  "expiration_date": null,
  "effective_date": null,
  "settlement_date": null,
  "offer_price": null,
  "conditions_restrictions": null,
  "processing_status": null,
  "account_number": null,
  "account_name": null,
  "eligible_quantity": null,
  "position_quantity": null,
  "cash_amount": null,
  "debit_credit_indicator": null,
  "audit": {
    "source_documents": [
      {
        "accession": "00011

## Summary: four JSON layers side-by-side


In [10]:
summary = pd.DataFrame([
    {
        'layer': 'A. EFTS JSON',
        'producer': 'SEC API',
        'example_keys': 'adsh, form, file_date, ciks, display_names, file_num',
        'covers_BNY_public_fields': '~10–15%',
        'shown_above': 'yes — live response',
    },
    {
        'layer': 'B. Filing HTML / exhibits',
        'producer': 'SEC Archives (raw docs)',
        'example_keys': 'HTML/text body, exhibits EX-99…',
        'covers_BNY_public_fields': 'most material facts (unstructured)',
        'shown_above': 'yes — raw HTML + text + keyword passages',
    },
    {
        'layer': 'C. Structured extraction JSON',
        'producer': 'Our model (to build)',
        'example_keys': 'options, dates, prices, conditions + source_passage',
        'covers_BNY_public_fields': 'target = ~all public fields',
        'shown_above': 'yes — empty template (nulls)',
    },
    {
        'layer': 'D. Client notification JSON',
        'producer': 'Our templates + BNY internals',
        'example_keys': 'A–C + account/position placeholders',
        'covers_BNY_public_fields': 'public + internal placeholders',
        'shown_above': 'yes — draft with null internals',
    },
])
display(summary)

print('\nArtifact paths for this run:')
print('  cached filing:', local if 'local' in dir() else None)
print('  document URL:', url if 'url' in dir() else None)


,layer,producer,example_keys,covers_BNY_public_fields,shown_above
0,A. EFTS JSON,SEC API,"adsh, form, file_date, ciks, display_names, file_num",~10–15%,yes — live response
1,B. Filing HTML / exhibits,SEC Archives (raw docs),"HTML/text body, exhibits EX-99…",most material facts (unstructured),yes — raw HTML + text + keyword passages
2,C. Structured extraction JSON,Our model (to build),"options, dates, prices, conditions + source_passage",target = ~all public fields,yes — empty template (nulls)
3,D. Client notification JSON,Our templates + BNY internals,A–C + account/position placeholders,public + internal placeholders,yes — draft with null internals



Artifact paths for this run:
  cached filing: /Users/kedarvaidya/Desktop/BNY-Capstone-Project/data/corpus/sample_docs/0001193125-20-011114__d874751dsctota.htm
  document URL: https://www.sec.gov/Archives/edgar/data/1492658/000119312520011114/d874751dsctota.htm
